# Strategy A: SCC-Block alphaP-WNS Drift Prior on GOLDEN

This notebook prototypes a block compromise: decompose the dynamic topology into SCCs, use alphaP-WNS inside each feedback SCC, and keep acyclic cross-SCC effects on the accepted GOLDEN scalar priors. Missing macro edges between SCCs remain exact zeros. Within an SCC, dense drift is allowed. This is a prior-level benchmark only.

In [1]:

from __future__ import annotations

import json
import time
from pathlib import Path

import networkx as nx
import numpy as np
from scipy.linalg import expm


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "apps" / "data-pipeline").exists() and (current / "data").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate repository root from notebook working directory")


REPO_ROOT = find_repo_root(Path.cwd())
GOLDEN_RUN = REPO_ROOT / "data" / ".private" / "GOLDEN" / "run"
STAGE1B_PATH = GOLDEN_RUN / "stage-1b.json"
MEGAPROMPT_PATH = GOLDEN_RUN / "stage-4-megaprompt.json"

stage1b = json.loads(STAGE1B_PATH.read_text())
megaprompt = json.loads(MEGAPROMPT_PATH.read_text())
accepted = megaprompt["accepted"]
causal_spec = stage1b["causal_spec"]
resolved_priors = {
    prior["parameter"]: prior
    for prior in accepted["resolved_priors"]
    if prior is not None
}

all_state_names = list(causal_spec["estimation"]["state_order"])
constructs = {construct["name"]: construct for construct in causal_spec["latent"]["constructs"]}
latent_names = [
    name
    for name in all_state_names
    if constructs[name].get("role") == "endogenous"
    and constructs[name].get("temporal_status") == "time_varying"
]
n_latent = len(latent_names)
latent_index = {name: idx for idx, name in enumerate(latent_names)}

drift_mask = np.eye(n_latent, dtype=bool)
for edge in causal_spec["latent"]["edges"]:
    cause = edge["cause"]
    effect = edge["effect"]
    parameter = f"beta_{cause}_{effect}"
    if cause in latent_index and effect in latent_index and parameter in resolved_priors:
        drift_mask[latent_index[effect], latent_index[cause]] = True

diag_mask = np.eye(n_latent, dtype=bool)
allowed_offdiag_mask = drift_mask & ~diag_mask
structural_zero_mask = (~drift_mask) & ~diag_mask
allowed_positions = [
    (row, col)
    for row in range(n_latent)
    for col in range(n_latent)
    if row != col and drift_mask[row, col]
]


def duration_to_days(value: str) -> float:
    if value.endswith("d"):
        return float(value[:-1])
    if value.endswith("h"):
        return float(value[:-1]) / 24.0
    raise ValueError(f"Unsupported model clock {value!r}")


model_dt_days = duration_to_days(causal_spec["measurement"]["model_clock"])

G = nx.DiGraph()
G.add_nodes_from(range(n_latent))
for row, col in allowed_positions:
    G.add_edge(col, row)
sccs = [tuple(sorted(component)) for component in nx.strongly_connected_components(G)]
sccs = sorted(sccs, key=lambda component: min(component))
component_index = {
    node: comp_idx
    for comp_idx, component in enumerate(sccs)
    for node in component
}
condensed = nx.DiGraph()
condensed.add_nodes_from(range(len(sccs)))
for row, col in allowed_positions:
    source_comp = component_index[col]
    target_comp = component_index[row]
    if source_comp != target_comp:
        condensed.add_edge(source_comp, target_comp)

macro_allowed_mask = np.eye(n_latent, dtype=bool)
for row in range(n_latent):
    for col in range(n_latent):
        if row == col:
            continue
        source_comp = component_index[col]
        target_comp = component_index[row]
        if source_comp == target_comp or condensed.has_edge(source_comp, target_comp):
            macro_allowed_mask[row, col] = True
macro_zero_mask = (~macro_allowed_mask) & ~diag_mask

print(f"Golden run: {GOLDEN_RUN.relative_to(REPO_ROOT)}")
print(f"Dynamic drift block ({n_latent} states): {', '.join(latent_names)}")
print("Excluded retained states:")
for name in all_state_names:
    if name not in latent_index:
        construct = constructs[name]
        print(f"  {name}: {construct.get('role')} / {construct.get('temporal_status')}")
print(f"Model interval: {model_dt_days:g} day")
print(f"Allowed off-diagonal dynamic drift entries: {int(allowed_offdiag_mask.sum())}")
print(f"Structural-zero off-diagonal dynamic entries: {int(structural_zero_mask.sum())}")
print("SCCs:")
for comp_idx, component in enumerate(sccs):
    labels = [latent_names[idx] for idx in component]
    print(f"  C{comp_idx}: {labels}")
print("Allowed dynamic topology edges:")
for row, col in allowed_positions:
    print(f"  {latent_names[col]} -> {latent_names[row]}")


Golden run: data/.private/GOLDEN/run
Dynamic drift block (9 states): sleep_quality, sleep_duration, screen_time, evening_screen_use, screen_content_type, social_media_use, stress, mental_health, bedtime_delay
Excluded retained states:
  chronotype: exogenous / time_invariant
Model interval: 1 day
Allowed off-diagonal dynamic drift entries: 11
Structural-zero off-diagonal dynamic entries: 61
SCCs:
  C0: ['sleep_quality']
  C1: ['sleep_duration']
  C2: ['screen_time']
  C3: ['evening_screen_use']
  C4: ['screen_content_type']
  C5: ['social_media_use']
  C6: ['stress', 'mental_health']
  C7: ['bedtime_delay']
Allowed dynamic topology edges:
  sleep_duration -> sleep_quality
  stress -> sleep_quality
  mental_health -> sleep_quality
  bedtime_delay -> sleep_duration
  stress -> screen_time
  mental_health -> screen_time
  screen_time -> evening_screen_use
  screen_content_type -> social_media_use
  mental_health -> stress
  stress -> mental_health
  evening_screen_use -> bedtime_delay


## Construction

Each singleton SCC keeps its accepted `rho_*` diagonal prior. Each multi-node SCC samples a local alphaP-WNS block, so stability of that diagonal block is guaranteed. Accepted scalar `beta_*` priors are then inserted only for edges between different SCCs. Since the SCC condensation graph is acyclic, these off-block couplings do not change the eigenvalues; the full drift matrix is stable when every SCC block is stable.

In [2]:

def sample_prior_1d(prior: dict, rng: np.random.Generator, n_draws: int) -> np.ndarray:
    family = prior["distribution"]
    params = prior["params"]
    if family == "Beta":
        return rng.beta(params["alpha"], params["beta"], size=n_draws)
    if family == "Uniform":
        return rng.uniform(params["lower"], params["upper"], size=n_draws)
    if family == "Normal":
        return rng.normal(params["mu"], params["sigma"], size=n_draws)
    raise ValueError(f"Unsupported drift prior family {family!r}")


def sample_current_scalar_prior(rng: np.random.Generator, n_draws: int) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    for idx, name in enumerate(latent_names):
        rho = sample_prior_1d(resolved_priors[f"rho_{name}"], rng, n_draws)
        rho = np.clip(rho, 1e-8, 1.0 - 1e-8)
        draws[:, idx, idx] = np.log(rho) / model_dt_days
    for row, col in allowed_positions:
        beta = sample_prior_1d(
            resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"],
            rng,
            n_draws,
        )
        draws[:, row, col] = beta / model_dt_days
    return draws


def interval_response_samples(draws: np.ndarray, dt_days: float, max_draws: int = 1000) -> np.ndarray:
    n = min(max_draws, draws.shape[0])
    return np.stack([expm(draws[i] * dt_days) for i in range(n)], axis=0)


def summarize_drift_samples(name: str, draws: np.ndarray, elapsed_seconds: float) -> dict[str, float | str]:
    eigvals = np.linalg.eigvals(draws)
    max_real = eigvals.real.max(axis=1)
    direct_absent_abs = np.abs(draws[:, structural_zero_mask])
    macro_absent_abs = np.abs(draws[:, macro_zero_mask])
    allowed_abs = np.abs(draws[:, allowed_offdiag_mask])
    response = interval_response_samples(draws, model_dt_days)
    direct_absent_response_abs = np.abs(response[:, structural_zero_mask])
    allowed_response_abs = np.abs(response[:, allowed_offdiag_mask])
    return {
        "name": name,
        "draws": draws.shape[0],
        "stable_rate": float(np.mean(max_real < 0.0)),
        "margin_q05": float(np.quantile(-max_real, 0.05)),
        "diag_mean": float(np.mean(np.diagonal(draws, axis1=1, axis2=2))),
        "direct_absent_a_q90": float(np.quantile(direct_absent_abs, 0.90)),
        "macro_absent_a_max": float(np.max(macro_absent_abs)),
        "allowed_a_q90": float(np.quantile(allowed_abs, 0.90)),
        "direct_absent_response_q90": float(np.quantile(direct_absent_response_abs, 0.90)),
        "allowed_response_q90": float(np.quantile(allowed_response_abs, 0.90)),
        "seconds": float(elapsed_seconds),
    }


def print_summary_table(rows: list[dict[str, float | str]]) -> None:
    columns = [
        ("name", "prior"),
        ("draws", "draws"),
        ("stable_rate", "stable"),
        ("margin_q05", "margin q05"),
        ("diag_mean", "diag mean"),
        ("direct_absent_a_q90", "direct absent |A| q90"),
        ("macro_absent_a_max", "macro absent |A| max"),
        ("allowed_a_q90", "allowed |A| q90"),
        ("direct_absent_response_q90", "direct absent |exp(AΔ)| q90"),
        ("allowed_response_q90", "allowed |exp(AΔ)| q90"),
        ("seconds", "seconds"),
    ]
    widths = []
    for key, label in columns:
        values = [label]
        for row in rows:
            value = row[key]
            values.append(str(value) if isinstance(value, str) else f"{value:.4g}")
        widths.append(max(len(v) for v in values))
    header = "  ".join(label.ljust(width) for (_, label), width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in rows:
        parts = []
        for (key, _label), width in zip(columns, widths):
            value = row[key]
            text = str(value) if isinstance(value, str) else f"{value:.4g}"
            parts.append(text.ljust(width))
        print("  ".join(parts))


In [3]:

def sample_wns_block(
    rng: np.random.Generator,
    n_draws: int,
    block_size: int,
    *,
    base_decay: float,
    skew_scale: float,
    wishart_df_extra: int = 8,
    wishart_mix: float = 0.20,
    alpha_shape: float = 16.0,
) -> np.ndarray:
    df = block_size + wishart_df_extra
    wishart_factors = rng.standard_normal((n_draws, df, block_size))
    wishart = np.einsum("bki,bkj->bij", wishart_factors, wishart_factors) / df
    p_inv = (1.0 - wishart_mix) * np.eye(block_size) + wishart_mix * wishart
    p_inv = p_inv + 1e-4 * np.eye(block_size)
    p = np.linalg.inv(p_inv)

    skew = np.zeros((n_draws, block_size, block_size), dtype=float)
    lower = np.tril_indices(block_size, k=-1)
    skew_values = rng.normal(0.0, skew_scale, size=(n_draws, len(lower[0])))
    skew[:, lower[0], lower[1]] = skew_values
    skew[:, lower[1], lower[0]] = -skew_values

    alpha = rng.gamma(shape=alpha_shape, scale=(2.0 * base_decay) / alpha_shape, size=n_draws)
    q_plus_s = alpha[:, None, None] * p + skew
    return -0.5 * np.einsum("bij,bjk->bik", p_inv, q_plus_s)


def sample_scc_block_wns_prior(rng: np.random.Generator, n_draws: int) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)

    scalar_reference = sample_current_scalar_prior(rng, min(n_draws, 2000))
    diagonal_decay_reference = -np.diagonal(scalar_reference, axis1=1, axis2=2)

    for component in sccs:
        component = tuple(component)
        if len(component) == 1:
            idx = component[0]
            rho = sample_prior_1d(resolved_priors[f"rho_{latent_names[idx]}"], rng, n_draws)
            rho = np.clip(rho, 1e-8, 1.0 - 1e-8)
            draws[:, idx, idx] = np.log(rho) / model_dt_days
            continue

        base_decay = float(np.median(diagonal_decay_reference[:, list(component)]))
        internal_betas = []
        for row in component:
            for col in component:
                if row == col:
                    continue
                prior = resolved_priors.get(f"beta_{latent_names[col]}_{latent_names[row]}")
                if prior is not None and prior["distribution"] in {"Normal", "Uniform"}:
                    params = prior["params"]
                    if prior["distribution"] == "Normal":
                        internal_betas.append(abs(float(params["mu"])))
                    else:
                        internal_betas.append(abs((float(params["lower"]) + float(params["upper"])) / 2.0))
        skew_scale = max(0.10, 1.5 * float(np.median(internal_betas or [0.10])))
        block_draws = sample_wns_block(
            rng,
            n_draws,
            len(component),
            base_decay=base_decay,
            skew_scale=skew_scale,
        )
        for local_row, global_row in enumerate(component):
            for local_col, global_col in enumerate(component):
                draws[:, global_row, global_col] = block_draws[:, local_row, local_col]

    # Acyclic cross-SCC effects keep the accepted scalar priors and exact topology.
    for row, col in allowed_positions:
        if component_index[row] == component_index[col]:
            continue
        beta = sample_prior_1d(
            resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"],
            rng,
            n_draws,
        )
        draws[:, row, col] = beta / model_dt_days

    return draws


## Benchmark

In [4]:

N_DRAWS = 5000
rng = np.random.default_rng(20260429)

start = time.perf_counter()
scalar_draws = sample_current_scalar_prior(rng, N_DRAWS)
scalar_elapsed = time.perf_counter() - start

start = time.perf_counter()
block_draws = sample_scc_block_wns_prior(rng, N_DRAWS)
block_elapsed = time.perf_counter() - start

rows = [
    summarize_drift_samples("accepted scalar prior", scalar_draws, scalar_elapsed),
    summarize_drift_samples("SCC-block alphaP-WNS", block_draws, block_elapsed),
]
print_summary_table(rows)


prior                  draws  stable  margin q05  diag mean  direct absent |A| q90  macro absent |A| max  allowed |A| q90  direct absent |exp(AΔ)| q90  allowed |exp(AΔ)| q90  seconds 
---------------------  -----  ------  ----------  ---------  ---------------------  --------------------  ---------------  ---------------------------  ---------------------  --------
accepted scalar prior  5000   1       3.603       -4.743     0                      0                     1.5              0.003527                     0.01428                0.001474
SCC-block alphaP-WNS   5000   1       2.908       -4.741     0                      0                     1.5              0.003039                     0.01817                0.005733


## Block Structure Check

In [5]:

print("Condensation edges:")
for source, target in condensed.edges():
    print(f"  C{source} -> C{target}")
print()
print(f"Macro-forbidden entries: {int(macro_zero_mask.sum())}")
print(f"Max |A| on macro-forbidden entries: {np.max(np.abs(block_draws[:, macro_zero_mask])):.6g}")
print(f"Direct-topology-forbidden entries: {int(structural_zero_mask.sum())}")
print(f"90% |A| on direct-forbidden entries: {np.quantile(np.abs(block_draws[:, structural_zero_mask]), 0.90):.6g}")


Condensation edges:
  C1 -> C0
  C2 -> C3
  C3 -> C7
  C4 -> C5
  C6 -> C0
  C6 -> C2
  C7 -> C1

Macro-forbidden entries: 61
Max |A| on macro-forbidden entries: 0
Direct-topology-forbidden entries: 61
90% |A| on direct-forbidden entries: 0


## Reading

This strategy keeps the SCC-level DAG exact and keeps stability almost-sure because the only non-WNS couplings are acyclic between stable diagonal blocks. It does not preserve direct zeros inside a feedback SCC; that is the intentional compromise. On this GOLDEN topology the only feedback SCC is the two-node stress/mental-health block, which already has both directed edges, so direct-zero leakage is not visible here.